In [24]:
import pandas as pd
import json
import langchain

In [25]:
df = pd.read_excel("Medical_list.xlsx")

In [26]:
# print(df.head())

In [27]:
chunks =[]
for index, row in df.iterrows():
    row_dict = row.to_dict()
    chunk_text = json.dumps(row_dict)
    chunks.append(chunk_text)

In [28]:
# print(chunks[0])

In [29]:
import os
from dotenv import load_dotenv
load_dotenv() 
api_key = os.getenv("GOOGLE_API_KEY")

In [30]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [31]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [32]:
from langchain_core.documents import Document

documents = [Document(page_content=chunk) for chunk in chunks]
document_ids = vector_store.add_documents(documents=documents)

# print(document_ids[:3])

In [33]:
# print(document_ids)

In [34]:
# retrieved_docs = vector_store.similarity_search("Give the cheapest Defibrillator in maharashtra?", k=3)

In [35]:
# retrieved_docs = vector_store.as_retriever(search_kwargs={"k": 3})
# resp = retrieved_docs.invoke("Give the cheapest Defibrillator in maharashtra?")
# print(resp)

In [36]:
# print(retrieved_docs)

In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="models/gemini-3.1-flash-lite",
    temperature=0.3
    
)

In [38]:
# question = input("Enter your question: ")
# retrieved_docs = vector_store.similarity_search(question, k=3)

In [39]:
# response = chain.invoke({"retrieved_docs": retrieved_docs, "question": question})

In [40]:
# print(response)

In [41]:
# print(retrieved_docs)

In [42]:
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage
from typing import List, Dict
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    completed: bool

memory = MemorySaver()  

In [43]:
state = State(messages=[], completed=False)

In [44]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)
prompt = ChatPromptTemplate.from_messages([   
    (
        "system",
        """
        You are a medical chatbot.
        Answer the user's question using the retrieved documents and the chat history only.
        Retrieved Documents:
        {retrieved_docs}
        """
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

In [45]:
chain = prompt | llm

In [46]:
from langchain_core.messages import AIMessage
from langchain_core.messages import HumanMessage


def chat(state:State):

    question = state["messages"][-1].content

    retrieved_docs = vector_store.similarity_search(
        question,
        k=3
    )

    response = chain.invoke({
        "retrieved_docs": retrieved_docs,
        "question": question,
        "chat_history": state["messages"]
    })

    return {
        "messages": [response]
    }

In [47]:
# chat(state)

In [48]:
from langgraph.graph import StateGraph, START, END


graph_builder = StateGraph(State)

graph_builder.add_node("chat", chat)

graph_builder.add_edge(START, "chat")
graph_builder.add_edge("chat", END)

In [49]:
chat_id = {}

In [59]:
import uuid
name = input("Enter your name: ")
new_or_not = input("New Chat? (yes/no): ").lower()

# NEW CHAT
if new_or_not == "yes":
    thread_id = str(uuid.uuid4())
    if name in chat_id:
        chat_id[name].append(thread_id)
    else:
        chat_id[name] = [thread_id]

# CONTINUE OLD CHAT
else:
    if name in chat_id:
        print("\nPrevious Chats:")
        for idx, tid in enumerate(chat_id[name]):
            print(f"{idx + 1}. {tid}")

        while True:
            try:
                choice = int(input("\nSelect chat number: "))
                if 1 <= choice <= len(chat_id[name]):
                    thread_id = chat_id[name][choice - 1]
                    break
                else:
                    print("Invalid choice.")
            except ValueError:
                print("Please enter a valid number.")


    else:
        print("No previous chat found.")
        thread_id = str(uuid.uuid4())
        chat_id[name] = [thread_id]
print(f"\nYour thread ID is: {thread_id}")


Previous Chats:
1. db82bcea-5d1a-43ab-bf4b-0ea3474d55ce
2. 837ce807-e2e1-42fe-84bc-4876a393349e

Your thread ID is: db82bcea-5d1a-43ab-bf4b-0ea3474d55ce


In [57]:
print(chat_id)

{'pushkar': ['db82bcea-5d1a-43ab-bf4b-0ea3474d55ce', '837ce807-e2e1-42fe-84bc-4876a393349e']}


In [52]:
graph = graph_builder.compile(
    checkpointer=memory
)

In [ ]:
config = {
    "configurable": {
        "thread_id": thread_id
    }
}
while True:

    question = input("\nUser: ")
    if question.lower() == "exit":
        break
    result = graph.invoke(
        {
            "messages": [
                HumanMessage(content=question)
            ]
        },
        config=config
    )
    print("\nUser:", question)
    print("\nAI:", result["messages"][-1].content[0]['text'])


User: what is my name

AI: Your name is Pushkar.


In [62]:
print(state['messages'])

[]


In [55]:
#Left out parts: adding the login and without login part memory
# UI for this 